# 02 - Student distillation

This notebook trains a smaller causal language model from the cached teacher logits. The hard target is the original response token, and the soft target is the teacher's top-k next-token distribution.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup


In [ ]:
@dataclass(frozen=True)
class StudentConfig:
    student_model: str = "distilgpt2"
    cache_dir: Path = Path("../artifacts/teacher_cache")
    output_dir: Path = Path("../checkpoints/student")
    report_dir: Path = Path("../reports")
    epochs: int = 1
    batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0
    temperature: float = 2.0
    distill_alpha: float = 0.65
    log_every: int = 10


config = StudentConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)
config.report_dir.mkdir(parents=True, exist_ok=True)
asdict(config)


In [ ]:
def select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def read_jsonl(path: Path) -> list[dict]:
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]


class TeacherCacheDataset(Dataset):
    def __init__(self, path: Path) -> None:
        self.records = read_jsonl(path)
        if not self.records:
            raise ValueError(f"empty teacher cache: {path}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        return self.records[index]


In [ ]:
device = select_device()
tokenizer = AutoTokenizer.from_pretrained(config.student_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

student = AutoModelForCausalLM.from_pretrained(config.student_model)
student.to(device)

{"device": str(device), "student_parameters": sum(p.numel() for p in student.parameters())}


In [ ]:
def collate_teacher_cache(rows: list[dict]) -> dict[str, torch.Tensor | list[str]]:
    max_length = max(len(row["input_ids"]) for row in rows)
    max_shift = max(len(row["loss_mask"]) for row in rows)
    top_k = len(rows[0]["teacher_topk_indices"][0])

    input_ids, attention_mask, loss_mask = [], [], []
    topk_indices, topk_logits = [], []
    for row in rows:
        pad_tokens = max_length - len(row["input_ids"])
        shift_pad = max_shift - len(row["loss_mask"])

        input_ids.append(row["input_ids"] + [tokenizer.pad_token_id] * pad_tokens)
        attention_mask.append(row["attention_mask"] + [0] * pad_tokens)
        loss_mask.append(row["loss_mask"] + [0] * shift_pad)
        topk_indices.append(row["teacher_topk_indices"] + [[0] * top_k for _ in range(shift_pad)])
        topk_logits.append(row["teacher_topk_logits"] + [[-1e4] * top_k for _ in range(shift_pad)])

    return {
        "ids": [row["id"] for row in rows],
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "loss_mask": torch.tensor(loss_mask, dtype=torch.bool),
        "teacher_topk_indices": torch.tensor(topk_indices, dtype=torch.long),
        "teacher_topk_logits": torch.tensor(topk_logits, dtype=torch.float32),
    }


train_loader = DataLoader(
    TeacherCacheDataset(config.cache_dir / "train_teacher_topk.jsonl"),
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_teacher_cache,
)
validation_loader = DataLoader(
    TeacherCacheDataset(config.cache_dir / "validation_teacher_topk.jsonl"),
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_teacher_cache,
)
len(train_loader), len(validation_loader)


In [ ]:
def move_batch(batch: dict, device: torch.device) -> dict:
    moved = {}
    for key, value in batch.items():
        moved[key] = value.to(device) if hasattr(value, "to") else value
    return moved


def masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return values[mask].mean()


def distillation_loss(model, batch: dict, temperature: float, alpha: float) -> dict[str, torch.Tensor]:
    outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    student_logits = outputs.logits[:, :-1, :]
    labels = batch["input_ids"][:, 1:]
    mask = batch["loss_mask"]

    hard_tokens = F.cross_entropy(
        student_logits.reshape(-1, student_logits.shape[-1]),
        labels.reshape(-1),
        reduction="none",
    ).reshape_as(labels)
    hard_loss = masked_mean(hard_tokens, mask)

    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    selected_student = torch.gather(student_log_probs, -1, batch["teacher_topk_indices"])
    teacher_probs = F.softmax(batch["teacher_topk_logits"] / temperature, dim=-1)
    soft_tokens = -(teacher_probs * selected_student).sum(dim=-1) * temperature**2
    soft_loss = masked_mean(soft_tokens, mask)

    total = alpha * soft_loss + (1.0 - alpha) * hard_loss
    return {"loss": total, "hard_loss": hard_loss.detach(), "soft_loss": soft_loss.detach()}


In [ ]:
sample_batch = move_batch(next(iter(train_loader)), device)
with torch.inference_mode():
    sample_metrics = distillation_loss(
        student,
        sample_batch,
        temperature=config.temperature,
        alpha=config.distill_alpha,
    )

{
    "input_shape": tuple(sample_batch["input_ids"].shape),
    "topk_shape": tuple(sample_batch["teacher_topk_indices"].shape),
    "supervised_tokens": int(sample_batch["loss_mask"].sum()),
    "initial_loss": float(sample_metrics["loss"]),
}


In [ ]:
optimizer = AdamW(student.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
update_steps = math.ceil(len(train_loader) / config.gradient_accumulation_steps) * config.epochs
warmup_steps = int(update_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=update_steps,
)
{"update_steps": update_steps, "warmup_steps": warmup_steps}


In [ ]:
@torch.inference_mode()
def evaluate_loader(model, loader: DataLoader) -> dict[str, float]:
    model.eval()
    totals = {"loss": 0.0, "hard_loss": 0.0, "soft_loss": 0.0, "batches": 0}
    for batch in loader:
        metrics = distillation_loss(
            model,
            move_batch(batch, device),
            temperature=config.temperature,
            alpha=config.distill_alpha,
        )
        for key in ("loss", "hard_loss", "soft_loss"):
            totals[key] += float(metrics[key])
        totals["batches"] += 1
    return {key: totals[key] / totals["batches"] for key in ("loss", "hard_loss", "soft_loss")}


In [ ]:
history = []
global_step = 0
student.train()
optimizer.zero_grad(set_to_none=True)

for epoch in range(config.epochs):
    progress = tqdm(train_loader, desc=f"epoch {epoch + 1}")
    for batch_index, batch in enumerate(progress, start=1):
        metrics = distillation_loss(
            student,
            move_batch(batch, device),
            temperature=config.temperature,
            alpha=config.distill_alpha,
        )
        (metrics["loss"] / config.gradient_accumulation_steps).backward()

        if batch_index % config.gradient_accumulation_steps == 0 or batch_index == len(train_loader):
            grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), config.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            row = {
                "epoch": epoch + 1,
                "step": global_step,
                "train_loss": float(metrics["loss"].detach()),
                "hard_loss": float(metrics["hard_loss"]),
                "soft_loss": float(metrics["soft_loss"]),
                "grad_norm": float(grad_norm),
                "lr": scheduler.get_last_lr()[0],
            }
            history.append(row)
            progress.set_postfix(loss=f"{row['train_loss']:.3f}")

    validation_metrics = evaluate_loader(student, validation_loader)
    history[-1].update({f"validation_{key}": value for key, value in validation_metrics.items()})

history[-3:]


In [ ]:
student.save_pretrained(config.output_dir)
tokenizer.save_pretrained(config.output_dir)

history_frame = pd.DataFrame(history)
history_frame.to_csv(config.report_dir / "distillation_history.csv", index=False)
(config.report_dir / "student_config.json").write_text(json.dumps(asdict(config), indent=2, default=str) + "\n")

history_frame.tail()


In [ ]:
if not history_frame.empty:
    ax = history_frame.plot(x="step", y=["train_loss", "hard_loss", "soft_loss"], figsize=(8, 4))
    ax.set_title("Student distillation losses")
    ax.set_ylabel("loss")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(config.report_dir / "distillation_losses.png", dpi=160)
